# Envelope Lifecycle Management

Envelopes are the savings containers that accumulate money for specific bill instances. This notebook explores the complete lifecycle of envelopes, from creation through balance tracking to funding status assessment.

**Topics covered:**
- Creating envelopes for bill instances
- Setting contribution windows
- Understanding initial allocations vs scheduled contributions
- Tracking balances over time
- Checking funding status


## Import Required Modules

Let's start by importing the necessary classes:


In [ ]:
from datetime import date
from decimal import Decimal
from sinkingfund import Bill, BillInstance, Envelope, CashFlow, CashFlowSchedule


## Creating Envelopes

An envelope is created for a specific `BillInstance`. It represents a dedicated savings container that will accumulate money toward that bill's payment.

**Required parameters:**
- `bill_instance`: The specific bill occurrence this envelope saves for
- Optional: `initial_allocation`: Starting balance (defaults to $0)
- Optional: `start_contrib_date`: When contributions can begin
- Optional: `end_contrib_date`: When contributions must end


In [ ]:
# First, create a bill instance.
bill_instance = BillInstance(
    bill_id="prop_tax",
    service="Property Tax 2025",
    due_date=date(2025, 11, 1),
    amount_due=Decimal("3600.00")
)

# Create an envelope with no initial allocation.
envelope = Envelope(
    bill_instance=bill_instance,
    initial_allocation=Decimal("0.00"),
    start_contrib_date=date(2025, 1, 1),
    end_contrib_date=date(2025, 10, 31)  # Day before due date
)

print(f"Envelope for: {envelope.bill_instance.service}")
print(f"Target amount: ${envelope.bill_instance.amount_due}")
print(f"Initial allocation: ${envelope.initial_allocation}")
print(f"Contribution window: {envelope.start_contrib_date} to {envelope.end_contrib_date}")
print(f"Due date: {envelope.bill_instance.due_date}")


### Envelopes with Initial Allocations

You can create an envelope with money already allocated (from an existing balance or previous savings):


In [ ]:
# Create an envelope with $1,000 already allocated.
envelope_with_allocation = Envelope(
    bill_instance=bill_instance,
    initial_allocation=Decimal("1000.00"),
    start_contrib_date=date(2025, 1, 1),
    end_contrib_date=date(2025, 10, 31)
)

print(f"Target amount: ${envelope_with_allocation.bill_instance.amount_due}")
print(f"Initial allocation: ${envelope_with_allocation.initial_allocation}")
print(f"Remaining needed: ${envelope_with_allocation.bill_instance.amount_due - envelope_with_allocation.initial_allocation}")


## Setting Contribution Windows

The contribution window defines when money can be added to an envelope. This is typically:
- **Start date**: Beginning of your planning period or when you start saving
- **End date**: The day before the bill's due date (or end of planning period)

This window ensures you stop contributing once the bill is due.


In [ ]:
# Bill due November 1st.
november_bill = BillInstance(
    bill_id="november_expense",
    service="November Expense",
    due_date=date(2025, 11, 1),
    amount_due=Decimal("2000.00")
)

# Envelope with 10-month contribution window.
envelope_10mo = Envelope(
    bill_instance=november_bill,
    initial_allocation=Decimal("0.00"),
    start_contrib_date=date(2025, 1, 1),  # January 1st
    end_contrib_date=date(2025, 10, 31)    # October 31st (day before due)
)

print(f"Bill due: {envelope_10mo.bill_instance.due_date}")
print(f"Contribution window: {envelope_10mo.start_contrib_date} to {envelope_10mo.end_contrib_date}")
print(f"Days to contribute: {(envelope_10mo.end_contrib_date - envelope_10mo.start_contrib_date).days} days")


## Understanding Initial Allocations vs Scheduled Contributions

Envelopes have two sources of funding:

1. **Initial Allocation**: A lump sum allocated when the envelope is created (e.g., from existing savings)
2. **Scheduled Contributions**: Regular payments added over time according to a contribution schedule

The total envelope balance = Initial Allocation + Scheduled Contributions (up to any given date)

Let's see this in action:


In [ ]:
# Create an envelope with initial allocation.
example_bill = BillInstance(
    bill_id="example_bill",
    service="Example Bill",
    due_date=date(2025, 6, 1),
    amount_due=Decimal("1200.00")
)

envelope = Envelope(
    bill_instance=example_bill,
    initial_allocation=Decimal("300.00"),  # $300 already saved
    start_contrib_date=date(2025, 1, 1),
    end_contrib_date=date(2025, 5, 31)
)

print("=== Envelope Configuration ===")
print(f"Target amount: ${envelope.bill_instance.amount_due}")
print(f"Initial allocation: ${envelope.initial_allocation}")
print(f"Still needed: ${envelope.bill_instance.amount_due - envelope.initial_allocation}")
print(f"Contribution window: {envelope.start_contrib_date} to {envelope.end_contrib_date}")

# At the start date, balance is just the initial allocation.
print(f"\nBalance on {envelope.start_contrib_date}: ${envelope.get_balance_as_of_date(envelope.start_contrib_date)}")
print("  (Only initial allocation, no contributions yet)")


## Tracking Balances Over Time

The `get_balance_as_of_date()` method calculates the envelope balance at any point in time by combining:
1. Initial allocation (always included)
2. Scheduled contributions up to that date

Let's create a schedule and see how the balance changes:


In [ ]:
# Create a contribution schedule (bi-weekly $150 contributions).
schedule = CashFlowSchedule()
contributions = [
    CashFlow(bill_id="example_bill", date=date(2025, 1, 15), amount=Decimal("150.00")),
    CashFlow(bill_id="example_bill", date=date(2025, 1, 29), amount=Decimal("150.00")),
    CashFlow(bill_id="example_bill", date=date(2025, 2, 12), amount=Decimal("150.00")),
    CashFlow(bill_id="example_bill", date=date(2025, 2, 26), amount=Decimal("150.00")),
    CashFlow(bill_id="example_bill", date=date(2025, 3, 12), amount=Decimal("150.00")),
    CashFlow(bill_id="example_bill", date=date(2025, 3, 26), amount=Decimal("150.00")),
]

schedule.add_cash_flows(contributions)
envelope.schedule = schedule

print("=== Balance Tracking Over Time ===")
print(f"Initial allocation: ${envelope.initial_allocation}")
print(f"Target amount: ${envelope.bill_instance.amount_due}\n")

# Check balance at various dates.
check_dates = [
    date(2025, 1, 1),   # Start
    date(2025, 1, 15),  # After first contribution
    date(2025, 2, 1),   # Mid-way through contributions
    date(2025, 3, 15),  # After several contributions
    date(2025, 5, 31),  # End of contribution window
    date(2025, 6, 1),   # Due date
]

for check_date in check_dates:
    balance = envelope.get_balance_as_of_date(check_date)
    contributions_to_date = schedule.total_amount_as_of_date(check_date)
    print(f"{check_date}: Balance = ${balance}")
    print(f"  (Initial: ${envelope.initial_allocation} + Contributions: ${contributions_to_date})")


## Checking Funding Status

The `remaining()` method calculates how much more is needed to fully fund the envelope by the due date. The `is_fully_funded()` method checks if the envelope will have enough by a specific date.


In [ ]:
# Create an envelope that needs more funding.
underfunded_bill = BillInstance(
    bill_id="underfunded",
    service="Underfunded Bill",
    due_date=date(2025, 12, 1),
    amount_due=Decimal("3000.00")
)

underfunded_envelope = Envelope(
    bill_instance=underfunded_bill,
    initial_allocation=Decimal("500.00"),
    start_contrib_date=date(2025, 1, 1),
    end_contrib_date=date(2025, 11, 30)
)

# Create a schedule with smaller contributions (might not be enough).
small_schedule = CashFlowSchedule()
small_contributions = [
    CashFlow(bill_id="underfunded", date=date(2025, 1, 15), amount=Decimal("200.00")),
    CashFlow(bill_id="underfunded", date=date(2025, 2, 15), amount=Decimal("200.00")),
    CashFlow(bill_id="underfunded", date=date(2025, 3, 15), amount=Decimal("200.00")),
]
small_schedule.add_cash_flows(small_contributions)
underfunded_envelope.schedule = small_schedule

# Check funding status at various points.
check_date = date(2025, 4, 1)
current = underfunded_envelope.get_balance_as_of_date(check_date)
remaining_needed = underfunded_envelope.remaining(check_date)
is_funded = underfunded_envelope.is_fully_funded(underfunded_bill.due_date)

print(f"=== Funding Status as of {check_date} ===")
print(f"Current balance: ${current}")
print(f"Target amount: ${underfunded_envelope.bill_instance.amount_due}")
print(f"Remaining needed: ${remaining_needed}")
print(f"Fully funded by due date ({underfunded_bill.due_date})? {is_funded}")


## Working with Contribution Schedules

A `CashFlowSchedule` contains the planned contributions for an envelope. You can:
- Add individual cash flows
- Query totals up to specific dates
- Get flows within date ranges

Let's explore schedule management:


In [ ]:
# Create a new envelope and schedule.
schedule_bill = BillInstance(
    bill_id="scheduled_bill",
    service="Scheduled Bill",
    due_date=date(2025, 6, 1),
    amount_due=Decimal("1800.00")
)

schedule_envelope = Envelope(
    bill_instance=schedule_bill,
    initial_allocation=Decimal("0.00"),
    start_contrib_date=date(2025, 1, 1),
    end_contrib_date=date(2025, 5, 31)
)

# Build a monthly contribution schedule.
monthly_schedule = CashFlowSchedule()
monthly_contribs = [
    CashFlow(bill_id="scheduled_bill", date=date(2025, 1, 15), amount=Decimal("360.00")),
    CashFlow(bill_id="scheduled_bill", date=date(2025, 2, 15), amount=Decimal("360.00")),
    CashFlow(bill_id="scheduled_bill", date=date(2025, 3, 15), amount=Decimal("360.00")),
    CashFlow(bill_id="scheduled_bill", date=date(2025, 4, 15), amount=Decimal("360.00")),
    CashFlow(bill_id="scheduled_bill", date=date(2025, 5, 15), amount=Decimal("360.00")),
]
monthly_schedule.add_cash_flows(monthly_contribs)
schedule_envelope.schedule = monthly_schedule

print("=== Schedule Analysis ===")
print(f"Total scheduled contributions: ${monthly_schedule.total_amount_as_of_date(date(2025, 12, 31))}")

# Get contributions in a specific range.
q1_contribs = monthly_schedule.cash_flows_in_range(
    start_date=date(2025, 1, 1),
    end_date=date(2025, 3, 31)
)
q1_total = monthly_schedule.total_amount_in_range(
    start_date=date(2025, 1, 1),
    end_date=date(2025, 3, 31)
)
print(f"\nQ1 contributions (Jan-Mar): ${q1_total}")
print(f"  Number of contributions: {len(q1_contribs)}")

# Check balance progression.
print("\n=== Balance Progression ===")
for contrib_date in [date(2025, 1, 15), date(2025, 3, 15), date(2025, 5, 15)]:
    balance = schedule_envelope.get_balance_as_of_date(contrib_date)
    print(f"{contrib_date}: ${balance}")


## Summary

**Key Concepts:**

1. **Envelope Creation**: Envelopes are created for specific `BillInstance` objects and represent savings containers.

2. **Initial Allocation**: One-time lump sum allocated when the envelope is created (from existing savings).

3. **Scheduled Contributions**: Regular payments added over time via a `CashFlowSchedule`.

4. **Contribution Windows**: Defined by `start_contrib_date` and `end_contrib_date` to control when contributions can occur.

5. **Balance Tracking**: `get_balance_as_of_date(date)` = Initial Allocation + Scheduled Contributions up to that date.

6. **Funding Status**: 
   - `remaining(date)` tells you how much more is needed
   - `is_fully_funded(date)` checks if the envelope will be fully funded by that date

**Next Steps:**
- Learn how to create contribution schedules in the CashFlow and CashFlowSchedule Patterns notebook
- See envelopes in action with allocation strategies
- Explore how envelopes work together in the SinkingFund workflow
